# Accompanying Code for "Strategies to Avoid Poor Local Minima in Nonlinear Optimization"
**Submission date**: 09.06.2026

**Author**: Nicolas Heidbrink

**Supervisor**: Stefan Schwarze, M. Sc.

**GitHub repository**: https://github.com/nicolasheidbrink/optimization-seminar


*Code structure, formatting, and documentation were optimized using Google Gemini.*

## 0. Imports

In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
from scipy.optimize import minimize
from contextlib import redirect_stdout
from tunneling_algorithm import TunnelingAlgorithm
import functions as fn
from algo_tester import Algorithm_Tester


## 1. The tunneling algorithm missing global minima
As mentioned in the seminar paper, the tunneling algorithm can stagnate when the movable pole gets closer and closer to the current iterate. This can be seen by examining the logs of an application of the tunneling function on the three-dimensional shubert function:


In [6]:
np.random.seed(42)
with open("shubert_3d_logs.txt", "w") as f:
    with redirect_stdout(f):
        ta = TunnelingAlgorithm(fn.shubert, fn.altered_shubert, [[-10, 10]]*3, verbose=True)
        x_stars, f_star = ta.apply_algorithm(np.array([np.random.uniform(-10, 10)]*3))
print(f"The tunneling algorithm found {len(x_stars)} points with the function value {f_star}")

The tunneling algorithm found 1 points with the function value -692.1324620094688


Between lines 693 and 1467, the iterates seem to stagnate around $(9.99, 6.78, 7.65)^T$. In the 10 iterations starting at line 692, the displacementsgo from being `[-0.00064652  0.01975908  0.01975908]` to `[-6.72446075e-07  1.93331515e-05  1.93331515e-05]`, equivalent to multiplication with $0.001040$, $0.000978$, and $0.000978$. According to the reasoning in the seminar paper, the expected factor would be $\frac{1}{2} ^ {10} = 0.000977$, which closely matches the observed results.

!!!Das funktioniert nicht mehr, der seed hat sich geändert oder so!!!

## 2. Improving the tunneling algorithm
To combat the problem of movable poles getting so close to tunneling phase iterates that the displacements gets extremely small, as mentioned in the seminar paper, one logical solution was to fix the movable pole's distance from iterates. Levy and Montalvo state that the movable pole should always be in the interior of a unit ball around the current iterate, so the distance was fixed to $0.9$. 

Also, a change was made to minimize the risk of random perturbations at the start of tunneling phases restricting the areas searched: These random perturbations now point generally in both directions down each axis of the domain. The changes were implemented in $\texttt{tunneling\_algorithm.py}$ and are activated by setting $\texttt{tunneling\_algorithm.improved}$ to $\texttt{True}$ while initializing the object.

The effect of the changes can be examined by testing both versions of the algorithm with $\texttt{algo\_tester.py}$, a script that runs the algorithms on a variety of functions and records whether and how many of their global minimal points were found in how much time.

In [7]:
algorithm_tester = Algorithm_Tester()
algorithm_tester.run_tests(TunnelingAlgorithm, improved=False)

Test: Shubert 2D, average number of minima found when successful: 1.85 out of 18, failure rate: 17.50% over 40 trials, time per trial: 0.402 seconds.
Test: Shubert 3D, average number of minima found when successful: 1.00 out of 81, failure rate: 60.00% over 10 trials, time per trial: 0.411 seconds.
Test: Shubert 6D, average number of minima found when successful: 0.00 out of 4374, failure rate: 100.00% over 10 trials, time per trial: 0.645 seconds.
Test: Six Hump Camel, average number of minima found when successful: 1.47 out of 2, failure rate: 0.00% over 30 trials, time per trial: 0.077 seconds.
Test: Altered Shubert 2D, average number of minima found when successful: 1.00 out of 1, failure rate: 86.67% over 30 trials, time per trial: 0.312 seconds.
Test: Function 15 2D, average number of minima found when successful: 1.00 out of 1, failure rate: 0.00% over 30 trials, time per trial: 0.125 seconds.
Test: Function 15 3D, average number of minima found when successful: 1.00 out of 1, f

In [8]:
algorithm_tester.run_tests(TunnelingAlgorithm, improved=True)

Test: Shubert 2D, average number of minima found when successful: 13.68 out of 18, failure rate: 0.00% over 40 trials, time per trial: 1.284 seconds.
Test: Shubert 3D, average number of minima found when successful: 13.30 out of 81, failure rate: 0.00% over 10 trials, time per trial: 0.938 seconds.
Test: Shubert 6D, average number of minima found when successful: 9.50 out of 4374, failure rate: 60.00% over 10 trials, time per trial: 0.723 seconds.
Test: Six Hump Camel, average number of minima found when successful: 2.00 out of 2, failure rate: 0.00% over 30 trials, time per trial: 0.099 seconds.
Test: Altered Shubert 2D, average number of minima found when successful: 1.00 out of 1, failure rate: 6.67% over 30 trials, time per trial: 0.305 seconds.
Test: Function 15 2D, average number of minima found when successful: 1.00 out of 1, failure rate: 0.00% over 30 trials, time per trial: 0.148 seconds.
Test: Function 15 3D, average number of minima found when successful: 1.00 out of 1, fai

Clearly, the results with the improvements are superior, although they still do not match Levy and Montalvo's results.

## 3. Implementing simulated annealing
As described in the seminar paper, Fast Simulated Annealing with an arbitrary initial temperature of 1000 was implemented under the flag `annealing` in `tunneling_algorithm.py`. The results are tested here.

In [9]:
algorithm_tester.run_tests(TunnelingAlgorithm, improved=True, annealing=True)

Test: Shubert 2D, average number of minima found when successful: 15.67 out of 18, failure rate: 10.00% over 40 trials, time per trial: 1.898 seconds.
Test: Shubert 3D, average number of minima found when successful: 11.20 out of 81, failure rate: 0.00% over 10 trials, time per trial: 0.979 seconds.
Test: Shubert 6D, average number of minima found when successful: 14.33 out of 4374, failure rate: 40.00% over 10 trials, time per trial: 1.950 seconds.
Test: Six Hump Camel, average number of minima found when successful: 1.00 out of 2, failure rate: 0.00% over 30 trials, time per trial: 0.654 seconds.
Test: Altered Shubert 2D, average number of minima found when successful: 1.00 out of 1, failure rate: 6.67% over 30 trials, time per trial: 0.718 seconds.
Test: Function 15 2D, average number of minima found when successful: 1.00 out of 1, failure rate: 10.00% over 30 trials, time per trial: 1.346 seconds.
Test: Function 15 3D, average number of minima found when successful: 1.00 out of 1, 